Código baseado no desenvolviemnto disponível em: https://www.kaggle.com/code/jeanmarques/rede-neural-para-imagens-termogr-ficas

In [3]:
from PIL import Image
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn.preprocessing import LabelBinarizer
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import RidgeClassifier
import xgboost as xgb
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import VGG19
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras import optimizers
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

In [4]:
#Leitura dos dados extraídos do repositório
tab = pd.read_csv("./dados/tabulares.csv", encoding='utf-8', low_memory=False)
caract = pd.read_csv("./dados/caracteristicas", encoding='utf-8', low_memory=False)
comp = pd.read_csv("./dados/complementares.csv", encoding='utf-8', low_memory=False)
#Junta dados
df = pd.merge(tab, caract, on='ID')
df = pd.merge(df, comp, on='ID')

In [5]:
df.head(3)

,ID,Records,Name,Age,Exams,Diagnosis,Data_registro,Estado_civil,Raca,Hábitos Alimentares,Temperatura,Menarca,Radioterapia,Cirurgia Plástica,Sinais
0,1,voluntária,private,72,thermography,unknown,2012-10-24,widow,pardo,Pobre em gordura,34.9,12,Não,Não,Não
1,2,77020,private,66,thermography,healthy,2012-10-30,married,pardo,Pobre em gordura,36.5,12,Não,Não,Não
2,3,voluntária,private,68,thermography,healthy,2012-10-30,single,black,Pobre em gordura,34.1,12,Não,Não,Não


In [6]:
#Traduzindo e padronizando algumas informações de características
df.Estado_civil = df.Estado_civil.str.replace('widow', 'viuva')
df.Estado_civil = df.Estado_civil.str.replace('married', 'casada')
df.Estado_civil = df.Estado_civil.str.replace('single', 'solteira')
df.Estado_civil = df.Estado_civil.str.replace('widow', 'viuva')
df.Estado_civil = df.Estado_civil.str.replace('viuvo', 'viuva')
df.Estado_civil = df.Estado_civil.str.replace('divorced', 'divorciada')
df.Estado_civil = df.Estado_civil.str.replace('casado', 'casada')
df.Estado_civil = df.Estado_civil.str.replace('divorciado', 'divorciada')
df.Estado_civil = df.Estado_civil.str.replace('solteiro', 'solteira')
df.Estado_civil = df.Estado_civil.str.replace('casada ', 'casada')
#
df.Raca = df.Raca.str.replace('black', 'negra')
df.Raca = df.Raca.str.replace('white', 'branca')
df.Raca = df.Raca.str.replace('mulattos', 'mulata')
df.Raca = df.Raca.str.replace('pardo', 'parda')

In [7]:
df.head(3)

,ID,Records,Name,Age,Exams,Diagnosis,Data_registro,Estado_civil,Raca,Hábitos Alimentares,Temperatura,Menarca,Radioterapia,Cirurgia Plástica,Sinais
0,1,voluntária,private,72,thermography,unknown,2012-10-24,viuva,parda,Pobre em gordura,34.9,12,Não,Não,Não
1,2,77020,private,66,thermography,healthy,2012-10-30,casada,parda,Pobre em gordura,36.5,12,Não,Não,Não
2,3,voluntária,private,68,thermography,healthy,2012-10-30,solteira,negra,Pobre em gordura,34.1,12,Não,Não,Não


In [8]:
#removendo dados irrelevantes para a análise e treinamento do modelo
lista_drop = ['Records', 'Name', 'Exams', 'Data_registro']
df = df.drop(lista_drop, axis=1)

In [9]:
#Retirando diagnósticos desconhecido
df_unknow = df.query(r"Diagnosis == 'unknown'")
df = df.query(r"Diagnosis != 'unknown'")
del df_unknow

In [10]:
df.head(3)

,ID,Age,Diagnosis,Estado_civil,Raca,Hábitos Alimentares,Temperatura,Menarca,Radioterapia,Cirurgia Plástica,Sinais
1,2,66,healthy,casada,parda,Pobre em gordura,36.5,12,Não,Não,Não
2,3,68,healthy,solteira,negra,Pobre em gordura,34.1,12,Não,Não,Não
3,4,66,healthy,casada,negra,Sem gordura,35.7,15,Não,Não,Não


In [11]:
#Função para converter imagem para array
def convert_image(image_to_convert):
  
  image_to_convert = image_to_convert.convert('RGB')
  #redimenciona imagem
  image_to_convert = image_to_convert.resize((224,224))
  
  return np.asarray(image_to_convert)

In [12]:
def select_images(patient):
  #Define array para receber imagens da paciente
  patient_images_array = []
  #Atribui caminho base para pastas de imagens
  images_directory = './dados/imagens/'
  #De acordo com o id da paciente, monta nome e pesquisa pela pasta dela
  patient_directory = (images_directory + 'PACIENTE_' + str(patient).zfill(4))
  #Veririca a quantidade de imagens dentro da pasta da paciente
  list_images = os.listdir(patient_directory) 
  for i, image in enumerate(list_images):
    #instancia imagem
    termography = Image.open(patient_directory + '/' + image)
    #Converte imagem
    converted_image = convert_image(termography)
    patient_images_array.append(converted_image)
  print('Paciente: ' + str(patient).zfill(4) + ' processado')
  return(patient_images_array)

In [13]:
df['Imagens'] = df.ID.apply(select_images)

Paciente: 0002 processado
Paciente: 0003 processado
Paciente: 0004 processado
Paciente: 0005 processado
Paciente: 0006 processado
Paciente: 0007 processado
Paciente: 0008 processado
Paciente: 0009 processado
Paciente: 0010 processado
Paciente: 0011 processado
Paciente: 0012 processado
Paciente: 0013 processado
Paciente: 0015 processado
Paciente: 0016 processado
Paciente: 0017 processado
Paciente: 0018 processado
Paciente: 0019 processado
Paciente: 0020 processado
Paciente: 0021 processado
Paciente: 0022 processado
Paciente: 0023 processado
Paciente: 0024 processado
Paciente: 0026 processado
Paciente: 0028 processado
Paciente: 0030 processado
Paciente: 0031 processado
Paciente: 0032 processado
Paciente: 0034 processado
Paciente: 0035 processado
Paciente: 0036 processado
Paciente: 0037 processado
Paciente: 0038 processado
Paciente: 0040 processado
Paciente: 0041 processado
Paciente: 0042 processado
Paciente: 0043 processado
Paciente: 0044 processado
Paciente: 0045 processado
Paciente: 00

In [14]:
#Verificando quantos pacientes estão saudáveis (healthy) e doentes (sick)
df.Diagnosis.value_counts()

Diagnosis
healthy    178
sick       100
Name: count, dtype: int64

In [ ]:
#Como temos duas categorias, atribui-se os rótulos 1 (sick) e 0 (healthy) 
lb = LabelBinarizer()
df['Diagnosis'] = lb.fit_transform(df['Diagnosis'])
df.Diagnosis.value_counts()

Diagnosis
0    178
1    100
Name: count, dtype: int64

In [16]:
print(df.Radioterapia.value_counts())
print(df.Sinais.value_counts())
df['Cirurgia Plástica'].value_counts()

Radioterapia
Não    235
Sim     43
Name: count, dtype: int64
Sinais
Não    206
Sim     72
Name: count, dtype: int64


Cirurgia Plástica
Não    257
Sim     21
Name: count, dtype: int64

In [ ]:
#Faz o mesmo com outros campos binários
# 0 - Não, 1 - Sim
df.Radioterapia = lb.fit_transform(df.Radioterapia)
# 0 - Não, 1 - Sim
df['Sinais'] = lb.fit_transform(df['Sinais'])
# 0 - Não, 1 - Sim
df['Cirurgia Plástica'] = lb.fit_transform(df['Cirurgia Plástica'])

print(df.Radioterapia.value_counts())
print(df.Sinais.value_counts())
df['Cirurgia Plástica'].value_counts()

Radioterapia
0    235
1     43
Name: count, dtype: int64
Sinais
0    206
1     72
Name: count, dtype: int64


Cirurgia Plástica
0    257
1     21
Name: count, dtype: int64

In [20]:
#Variáveis não binárias

print(df['Estado_civil'].value_counts())
print(df['Raca'].value_counts())
print(df['Hábitos Alimentares'].value_counts())


Estado_civil
casada        125
solteira       95
viuva          37
divorciada     21
Name: count, dtype: int64
Raca
branca      101
parda        85
negra        77
amarela      13
indigena      1
mulata        1
Name: count, dtype: int64
Hábitos Alimentares
Pobre em gordura    214
Rica em gordura      43
Sem gordura          21
Name: count, dtype: int64


In [18]:
#Faz o mesmo com dados categóricos que não são binários
l_encoder = preprocessing.LabelEncoder()

# 0 - casada; 1 - divorciada; 2 - solteira; 3 - viúva
df['Estado_civil'] = l_encoder.fit_transform(df['Estado_civil'])
#0 - amarela; 1 - branca; 2 - indígena; 3 - mulata; 4 - negra; 5 - parda
df['Raca'] = l_encoder.fit_transform(df['Raca'])
#0 - pobre em gordura; 1 - rico em gordura; 2 - sem gordura
df['Hábitos Alimentares'] = l_encoder.fit_transform(df['Hábitos Alimentares'])

print(df['Estado_civil'].value_counts())
print(df['Raca'].value_counts())
print(df['Hábitos Alimentares'].value_counts())

Estado_civil
0    125
2     95
3     37
1     21
Name: count, dtype: int64
Raca
1    101
5     85
4     77
0     13
2      1
3      1
Name: count, dtype: int64
Hábitos Alimentares
0    214
1     43
2     21
Name: count, dtype: int64


In [19]:
def separa_df(dataframe):
  imagens = []
  diagnostico = []
  for i in range(dataframe.shape[0]):
    for imagem in (dataframe.iloc[i,1]):
        imagens.append(imagem)
        diagnostico.append(dataframe.iloc[i,0])
      
  return(imagens, diagnostico) 

In [20]:
#cria novo dataframe apenas com as imagens e os diagnósticos
new_df = df[['Diagnosis','Imagens']]

imagens, diagnosticos = separa_df(new_df)

imagens = np.array(imagens)
diagnosticos = np.array(diagnosticos)

In [21]:
#verificando se os tamnhos coincidem 
len(imagens), len(diagnosticos)

(7245, 7245)

In [22]:
diagnosticos

array([0, 0, 0, ..., 1, 1, 1])

In [23]:
#CALLBACKS
#São funções que podem ser passadas nos métodos do Keras para se integrar e auxiliar várias etapas do ciclo de vida do treinamento
#ModelCheckpoint: serve para salvar o modelo com uma certa frequência. Útil caso ocorra algum problema durante o processamento.
#https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/ModelCheckpoint
#ReduceLROnPlateau: Reduz a taxa de aprendizado quando alguma métrica para de melhorar com o avanço do treinamento. Muitos modelos se beneficiam ao reduzer a taa de aprendizado em 20%

from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau


In [24]:
#Definindo hiperparâmetros do modelo
batch_size   = 32
#Tamanho da imagem
input_shape  = (224, 224, 3)
#
random_state = 42
delta        = 1e-5
epoch        = 50

In [40]:
'''
path: onde será salvo o modelo
monitor: métrica a ser monitorada (acurácia)
verbose: (1) mostra na barra de progresso
save_best_only: Salvar somente o melhor modelo
mode: Queremos a máxima acurácia.
'''
###
path='transferlearning_weights.keras'
checkpoint = ModelCheckpoint(path, monitor='accuracy', verbose=1, save_best_only=True, mode='max')

In [41]:

#monitor: valor observado (acurácia)
#factor: fator de redução
#min-delta: limiar para medir o novo ótimo, para focar apenas em mudanças significativas
#patience: número de épocas sem evolução do modelo
#verbose = 1: mostra mensagens de atualização
lr_reduce = ReduceLROnPlateau(monitor='accuracy', factor=0.2, min_delta=delta, patience=5, verbose=1)

callbacks = [checkpoint, lr_reduce]

In [27]:
#Separa em treino e teste numa proporção de 70/30
(x_train, x_test, y_train, y_test) = train_test_split(imagens, diagnosticos, test_size=0.30, stratify=diagnosticos, random_state=random_state)

In [28]:
y_train = to_categorical(y_train)
y_test = to_categorical(y_test)

In [29]:
from tensorflow.keras.applications.vgg19 import preprocess_input

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    zoom_range=0.1
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

data_aug = val_datagen.flow(x_test, y_test, batch_size=batch_size)

In [69]:
#Data augmentation
#gera mais diversidade de dados, sem necessidade de fazer novas coletas
#train_datagen = ImageDataGenerator(
#        rotation_range=10,
#        zoom_range=0.1)

#data_aug = train_datagen.flow(x_train, y_train, batch_size=batch_size)

In [30]:
#transfer learning
#aproveita uma modelo treinado anteriormente
conv_base = VGG19(weights='imagenet', include_top=False, input_shape=input_shape)
conv_base.summary()

Model: "vgg19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv4 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv4 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv4 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,024,384 (76.39 MB)

 Trainable params: 20,024,384 (76.39 MB)

 Non-trainable params: 0 (0.00 B)

In [31]:
#Retreinando parte da VGG19
conv_base.trainable = True
set_trainable = False
for layer in conv_base.layers:
  if layer.name == 'block5_conv1':
    set_trainable = True
  if set_trainable:
    layer.trainable = True
  else:
    layer.trainable = False
#

conv_base.summary()

Model: "vgg19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv4 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv4 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv4 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,024,384 (76.39 MB)

 Trainable params: 9,439,232 (36.01 MB)

 Non-trainable params: 10,585,152 (40.38 MB)

In [37]:
#Instanciando modelo
    
model = models.Sequential()
model.add(conv_base)
model.add(layers.GlobalAveragePooling2D()) #Pooling redução de resolução
model.add(layers.BatchNormalization()) #Adiciona normalização
#model.add(layers.Flatten())
model.add(layers.Dense(128, activation='relu'))
model.add(layers.Dropout(0.4))
model.add(layers.Dense(2, activation='softmax'))
#

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ vgg19 (Functional)              │ (None, 7, 7, 512)      │    20,024,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ ?                      │   0 (unbuilt) │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,024,384 (76.39 MB)

 Trainable params: 9,439,232 (36.01 MB)

 Non-trainable params: 10,585,152 (40.38 MB)

In [38]:
model.compile(loss='categorical_crossentropy',
                  optimizer='adam',
                  metrics=["accuracy"])

In [46]:
history = model.fit(
    data_aug,
    validation_data=(x_test, y_test),
    callbacks=callbacks,
    epochs=epoch
)

Epoch 1/50
68/68 ━━━━━━━━━━━━━━━━━━━━ 0s 9s/step - accuracy: 0.9598 - loss: 0.1224

KeyboardInterrupt: 

In [45]:
history = model.fit(data_aug,
                    steps_per_epoch=len(x_train)// batch_size, # parte inteira da divisão
                    validation_data=(x_test, y_test),
                    validation_steps=len(x_test) // batch_size,# parte inteira da divisão
                    callbacks=callbacks,
                    epochs=epoch)

Epoch 1/50
 68/158 ━━━━━━━━━━━━━━━━━━━━ 11:41 8s/step - accuracy: 0.9591 - loss: 0.1182

C:\Users\luiz.altimari\AppData\Local\Programs\Python\Python311\Lib\contextlib.py:155: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self.gen.throw(typ, value, traceback)


KeyboardInterrupt: 